# 📖 Notebook 1: RPO & RTO Fundamentals

## Why This Matters

Imagine you run an online store. Your database crashes at 2:00 PM.
Your last backup was at midnight. That means you just **lost 14 hours of
orders, payments, and customer data**.

How much money did you lose? How many customers are furious?

This is exactly why enterprises like Microsoft, JPMorgan, and hospitals
spend millions on **Business Continuity and Disaster Recovery (BCDR)**.

In this notebook, you will learn the two most important numbers in BCDR:
- **RPO** (Recovery Point Objective) — how much data you can afford to lose
- **RTO** (Recovery Time Objective) — how long you can afford to be down

## Learning Objectives

By the end of this notebook, you will understand:
- The difference between RPO and RTO
- How to calculate the business impact of downtime
- SLA math (what "five nines" really means)
- How different standby types affect RPO and RTO
- How to choose the right strategy for your system

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 08-enterprise/bcdr
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it does not appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import time
import datetime
from tabulate import tabulate

# Database connection settings
DB_PRIMARY = {
    "host": "localhost",
    "port": 5432,
    "database": "bcdr_demo",
    "user": "demo",
    "password": "demo"
}

DB_STANDBY = {
    "host": "localhost",
    "port": 55433,
    "database": "bcdr_demo",
    "user": "demo",
    "password": "demo"
}

def get_primary_connection():
    return psycopg2.connect(**DB_PRIMARY)

def get_standby_connection():
    return psycopg2.connect(**DB_STANDBY)

# Test connection
try:
    conn = get_primary_connection()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM orders")
    count = cur.fetchone()[0]
    print(f"✅ Connected to primary. Found {count} orders in the database.")
    conn.close()
except Exception as e:
    print(f"❌ Could not connect to primary: {e}")
    print("Make sure docker-compose is running: docker compose up -d")

## 📚 What is RPO? (Recovery Point Objective)

**RPO answers: "How much data can we afford to lose?"**

Think of RPO as looking **backward in time** from the disaster:

```
  ──────[last backup]──────────[new data written]──────[💥 DISASTER]──────
                        ▲                                    ▲
                        │◄──── This data is LOST ──────────►│
                        │         (this is RPO)              │
```

### Real-World RPO Examples

| System | RPO | Why |
|--------|-----|-----|
| Stock Exchange | ~0 seconds | Every trade is legally binding |
| Banking | < 1 second | Financial regulations require it |
| Hospital Records | < 1 minute | Patient safety depends on it |
| E-commerce | < 5 minutes | Losing orders = losing revenue |
| Blog | < 24 hours | Content can be rewritten |
| Dev/Test | Days | No real users affected |

### Key Insight
**Lower RPO = more expensive.** Near-zero RPO requires synchronous replication, which adds latency to every write.

In [ ]:
# =============================================================================
# Demo: Visualizing Data Loss at Different RPO Levels
# =============================================================================
# RPO is a *time window*, not a row count. What the business feels is the rows
# that fall inside that window, so we convert the window into orders and
# dollars using the real write rate of this database.

conn = get_primary_connection()
cur = conn.cursor()

cur.execute(
    "SELECT COUNT(*) as total_orders, "
    "SUM(total_amount) as total_revenue, "
    "AVG(total_amount) as avg_order_value "
    "FROM orders"
)
total_orders, total_revenue, avg_order = cur.fetchone()
conn.close()

# Simulate order rate: assume our 500 orders happened over 30 days
orders_per_hour = total_orders / (30 * 24)
revenue_per_hour = float(total_revenue) / (30 * 24)

# The seeded lab database is deliberately tiny -- well under one order an
# hour. At that rate a 5-minute RPO and a 0-second RPO both round to "no
# orders lost", which teaches nothing, so we also show the same ladder at a
# rate a real store of this shape would run at. Same maths, honest labels.
PROD_ORDERS_PER_HOUR = 1200.0
AVG_ORDER = float(avg_order)

print("=" * 84)
print("DATA LOSS ANALYSIS AT DIFFERENT RPO LEVELS")
print("=" * 84)
print(f"\nLab database: {total_orders} orders, ${total_revenue:,.2f} total revenue")
print(f"Lab rate:     {orders_per_hour:.2f} orders/hour, ${revenue_per_hour:,.2f}/hour")
print(f"Prod rate:    {PROD_ORDERS_PER_HOUR:,.0f} orders/hour, "
      f"${PROD_ORDERS_PER_HOUR * AVG_ORDER:,.2f}/hour\n")

rpo_scenarios = [
    ("24 hours (daily backup)",      24),
    ("1 hour (hourly backup)",        1),
    ("5 minutes (async streaming)", 5/60),
    ("0 seconds (sync replication)",  0),
]

table = []
lab_losses = []
for label, hours in rpo_scenarios:
    lab_orders = orders_per_hour * hours
    lab_losses.append(lab_orders)
    table.append([
        label,
        f"{lab_orders:,.2f}",
        f"${revenue_per_hour * hours:,.2f}",
        f"{PROD_ORDERS_PER_HOUR * hours:,.0f}",
        f"${PROD_ORDERS_PER_HOUR * hours * AVG_ORDER:,.0f}",
    ])

print(tabulate(table,
    headers=["RPO window", "Lab orders lost", "Lab $ lost",
             "Prod orders lost", "Prod $ lost"],
    tablefmt="grid"))

print(f"\n💡 A daily backup puts {orders_per_hour * 24:,.1f} orders "
      f"(${revenue_per_hour * 24:,.2f}) at risk in THIS database -- and "
      f"{PROD_ORDERS_PER_HOUR * 24:,.0f} orders "
      f"(${PROD_ORDERS_PER_HOUR * 24 * AVG_ORDER:,.0f}) at production rate.")
print("   Each step down the ladder costs more to run and saves more when it fires.")

# A wider RPO window can never lose LESS data than a narrower one, and a
# zero-second RPO loses nothing by definition. If either stops holding, the
# arithmetic above has drifted.
assert lab_losses == sorted(lab_losses, reverse=True), (
    f"RPO ladder is not monotonic: {lab_losses}")
assert lab_losses[-1] == 0, "a zero-second RPO must lose nothing by definition"


## 📚 What is RTO? (Recovery Time Objective)

**RTO answers: "How long can we be down?"**

Think of RTO as looking **forward in time** from the disaster:

```
  ──────[💥 DISASTER]──────────[recovery work]──────[✅ BACK ONLINE]──────
              ▲                                            ▲
              │◄──────── System is DOWN ──────────────────►│
              │              (this is RTO)                 │
```

### Real-World RTO Examples

| System | RTO | Why |
|--------|-----|-----|
| Payment Processing | < 30 seconds | Every second = lost transactions |
| Hospital Systems | < 5 minutes | Patient care cannot wait |
| Banking | < 15 minutes | Regulatory requirement |
| E-commerce | < 1 hour | Customer trust + revenue |
| Internal Tools | < 4 hours | Employees can do other work |
| Reporting | < 24 hours | Reports can wait |

### Key Insight
**Lower RTO = more expensive.** A 30-second RTO requires hot standby servers running 24/7, automatic failover, and extensive testing.

In [ ]:
# =============================================================================
# Demo: Cost of Downtime Calculator
# =============================================================================
# RTO is the OTHER number, and it is independent of RPO: you can have a
# 0-second RPO (lost nothing) and still be down for four hours, or lose an
# hour of writes and be back up in ten seconds. RPO prices the data you lost;
# RTO prices the time you were unavailable.

ORDERS_PER_HOUR = orders_per_hour
AVG_ORDER_VALUE = float(avg_order)
SUPPORT_STAFF_HOURLY = 50
SUPPORT_STAFF_COUNT = 5
SLA_PENALTY_PER_HOUR = 1000
REPUTATION_COST_PER_HOUR = 500

print("=" * 88)
print("COST OF DOWNTIME ANALYSIS")
print("=" * 88)

rto_scenarios = [
    ("30 seconds",  0.5/60),
    ("5 minutes",   5/60),
    ("1 hour",      1),
    ("4 hours",     4),
    ("24 hours",    24),
]

table = []
totals = []
for label, hours in rto_scenarios:
    lost_revenue = ORDERS_PER_HOUR * AVG_ORDER_VALUE * hours
    # Staff cost is floored at one hour: once you page the on-call bridge you
    # pay for the hour even if the outage itself lasted thirty seconds.
    staff_cost = SUPPORT_STAFF_HOURLY * SUPPORT_STAFF_COUNT * max(hours, 1)
    sla_penalty = SLA_PENALTY_PER_HOUR * hours
    reputation = REPUTATION_COST_PER_HOUR * hours
    total = lost_revenue + staff_cost + sla_penalty + reputation
    totals.append(total)
    table.append([label, f"${lost_revenue:,.0f}", f"${staff_cost:,.0f}",
                  f"${sla_penalty:,.0f}", f"${reputation:,.0f}", f"${total:,.0f}"])

print(f"\nBusiness parameters:")
print(f"  Order rate:  {ORDERS_PER_HOUR:.2f}/hour x ${AVG_ORDER_VALUE:,.2f} avg")
print(f"  Support:     {SUPPORT_STAFF_COUNT} staff x ${SUPPORT_STAFF_HOURLY}/hour "
      f"(billed at a one-hour minimum)")
print(f"  SLA penalty: ${SLA_PENALTY_PER_HOUR}/hour")
print(f"  Reputation:  ${REPUTATION_COST_PER_HOUR}/hour\n")

print(tabulate(table,
    headers=["Downtime (RTO)", "Lost Revenue", "Staff Cost", "SLA Penalty",
             "Reputation", "TOTAL Cost"],
    tablefmt="grid"))
print("\n💡 This is why enterprises invest millions in reducing RTO!")

# Every component is shown, so the columns must add up to TOTAL, and a longer
# outage must never come out cheaper than a shorter one.
h = 4
expected = (ORDERS_PER_HOUR * AVG_ORDER_VALUE * h
            + SUPPORT_STAFF_HOURLY * SUPPORT_STAFF_COUNT * h
            + SLA_PENALTY_PER_HOUR * h
            + REPUTATION_COST_PER_HOUR * h)
assert abs(expected - totals[3]) < 1e-6, (
    f"the columns do not sum to TOTAL: {expected} vs {totals[3]}")
assert totals == sorted(totals), f"cost is not monotonic in downtime: {totals}"


## 📚 SLA Math — What "Five Nines" Really Means

You will often hear availability described as "nines":

| SLA | Uptime % | Downtime per Year | Downtime per Month |
|-----|----------|-------------------|-------------------|
| Two nines | 99% | 3.65 days | 7.3 hours |
| Three nines | 99.9% | 8.76 hours | 43.8 minutes |
| Four nines | 99.99% | 52.6 minutes | 4.38 minutes |
| Five nines | 99.999% | 5.26 minutes | 26.3 seconds |

Each additional "nine" is exponentially harder and more expensive to achieve.

In [ ]:
# =============================================================================
# Demo: SLA Calculator
# =============================================================================

print("=" * 65)
print("SLA AVAILABILITY CALCULATOR")
print("=" * 65)

sla_levels = [
    ("Two nines",   0.99),
    ("Three nines", 0.999),
    ("Four nines",  0.9999),
    ("Five nines",  0.99999),
    ("Six nines",   0.999999),
]

MINUTES_PER_YEAR = 365.25 * 24 * 60
MINUTES_PER_MONTH = 30.44 * 24 * 60

table = []
for name, avail in sla_levels:
    down_yr = MINUTES_PER_YEAR * (1 - avail)
    down_mo = MINUTES_PER_MONTH * (1 - avail)

    if down_yr >= 1440:
        yr_str = f"{down_yr/1440:.1f} days"
    elif down_yr >= 60:
        yr_str = f"{down_yr/60:.1f} hours"
    else:
        yr_str = f"{down_yr:.1f} min"

    if down_mo >= 60:
        mo_str = f"{down_mo/60:.1f} hours"
    elif down_mo >= 1:
        mo_str = f"{down_mo:.1f} min"
    else:
        mo_str = f"{down_mo*60:.1f} sec"

    table.append([name, f"{avail*100:.4f}%", yr_str, mo_str])

print()
print(tabulate(table,
    headers=["SLA Level", "Availability", "Downtime/Year", "Downtime/Month"],
    tablefmt="grid"))
print()
print("💡 Going from 99.9% to 99.99% = allowed downtime drops by 10x!")
print("   Requires hot standby, automatic failover, and multi-region.")

## 📚 Hot, Warm, and Cold Standby

The standby type determines both your RPO and RTO:

### Hot Standby (what we use in this lab)
- Server is **running and receiving real-time data updates**
- Can serve read-only queries while standing by
- Failover takes **seconds to minutes**
- **Example**: PostgreSQL streaming replication, Redis replica

### Warm Standby
- Server is **running but data is synced periodically** (e.g., every hour)
- Needs to "catch up" before it can become primary
- Failover takes **minutes to hours**
- **Example**: Periodic database snapshots restored to standby

### Cold Standby
- Server **exists but is turned off** (or does not exist yet)
- Must be started, then restore from backup
- Failover takes **hours to days**
- **Example**: AWS AMI that you launch and restore a backup to

In [ ]:
# =============================================================================
# Demo: Check Our Hot Standby Replication Status
# =============================================================================

conn = get_primary_connection()
cur = conn.cursor()

cur.execute(
    "SELECT client_addr, state, sent_lsn, write_lsn, "
    "flush_lsn, replay_lsn, sync_state "
    "FROM pg_stat_replication"
)
rows = cur.fetchall()
conn.close()

if rows:
    print("=" * 65)
    print("REPLICATION STATUS (from primary)")
    print("=" * 65)
    for row in rows:
        print(f"  Standby address:  {row[0]}")
        print(f"  State:            {row[1]}")
        print(f"  Sent LSN:         {row[2]}")
        print(f"  Written LSN:      {row[3]}")
        print(f"  Flushed LSN:      {row[4]}")
        print(f"  Replayed LSN:     {row[5]}")
        print(f"  Sync mode:        {row[6]}")
    print()
    print("💡 async = writes don't wait for standby. Better perf but RPO > 0.")
    print("   sync = RPO 0, but every write waits for standby confirmation.")
else:
    print("⚠️  No standby connected.")
    print("   Run: docker compose up -d pg-standby")

In [ ]:
# =============================================================================
# Demo: Measure Replication Lag in Real-Time
# =============================================================================
# Replication lag is what BOUNDS your achievable RPO. With asynchronous
# replication the primary acknowledges a commit before the standby has it, so
# if the primary dies right now everything written inside the lag window is
# gone. Your RPO can never be tighter than your typical lag.

primary_conn = get_primary_connection()
primary_conn.autocommit = True
primary_cur = primary_conn.cursor()

standby_conn = get_standby_connection()
standby_cur = standby_conn.cursor()

print("=" * 65)
print("MEASURING REPLICATION LAG")
print("=" * 65)
print("Writing a test record to primary, timing standby visibility...\n")

POLL_INTERVAL = 0.005   # the measurement below is quantised by this
TIMEOUT = 10.0

marker = f"RPO_TEST_{int(time.time())}"
write_start = time.time()

primary_cur.execute(
    "INSERT INTO audit_log (table_name, record_id, action, changed_by) "
    "VALUES (%s, %s, %s, %s)",
    ('rpo_test', 0, 'INSERT', marker)
)
write_time = time.time() - write_start

# Poll the standby until it sees the record.
poll_start = time.time()
lag = None
attempts = 0

while time.time() - poll_start < TIMEOUT:
    attempts += 1
    try:
        standby_conn.rollback()  # end the snapshot so we can see new commits
        standby_cur.execute(
            "SELECT id FROM audit_log WHERE changed_by = %s", (marker,)
        )
        if standby_cur.fetchone():
            lag = time.time() - poll_start
            break
    except psycopg2.Error:
        pass
    time.sleep(POLL_INTERVAL)

# Clean up before asserting, so a failure does not leave the marker behind.
primary_cur.execute("DELETE FROM audit_log WHERE changed_by = %s", (marker,))
primary_conn.close()
standby_conn.close()

assert lag is not None, (
    f"the standby never saw the write within {TIMEOUT}s -- streaming "
    f"replication is broken, and nothing in this lab can demonstrate a "
    f"bounded RPO without it")

print(f"  ✅ Commit on primary:   {write_time*1000:.1f} ms")
print(f"  ✅ Visible on standby:  {lag*1000:.1f} ms "
      f"({attempts} poll(s), +/-{POLL_INTERVAL*1000:.0f} ms quantisation)")
print(f"\n  Measured replication lag: ~{lag*1000:.0f} ms")
print("  -> ASYNC: that lag is the upper bound on how fresh the standby can be,")
print("     so it is also the tightest RPO you can honestly promise.")
print("  -> SYNC:  the commit itself waits for the standby, so RPO is 0 and the")
print("     write latency above absorbs the lag instead. That is the trade.")

# A hot standby on a local stack replicates in milliseconds. Seconds of lag
# means this is no longer a hot standby, and the RPO claim above is fiction.
assert lag < 5.0, (
    f"replication lag of {lag:.1f}s is far too high for a hot standby -- "
    f"check `pg_stat_replication` on the primary")


## 📝 Summary

### What You Learned

1. **RPO** — How much data you can lose. Lower RPO = faster replication = more cost.
2. **RTO** — How long you can be down. Lower RTO = hot standby + automation = more cost.
3. **SLA Math** — Each nine is 10x harder. Five nines = 5.26 min downtime/year.
4. **Standby Types** — Hot (seconds), Warm (minutes), Cold (hours).
5. **Business Impact** — Lost revenue + SLA penalties + staff costs + reputation.

### Key Takeaway

> **BCDR is not about preventing failures — failures WILL happen.**
> **BCDR is about how quickly and completely you recover.**

### Next Notebook

In **Notebook 2**, we dive into database replication — how PostgreSQL
streaming replication works, and how to perform a failover.